<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/Grupo7/Volume_de_Credito_e_Derivativos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install wbdata

import wbdata
import pandas as pd
from IPython.display import display, Markdown
from google.colab import files


# 2. DEFINIR OS PAÍSES E OS ANOS

paises = ["ITA", "FRA", "GBR", "CHN"]

ano_inicial = "2000"
ano_final = "2025"


# 3. BUSCAR OS DADOS DE CRÉDITO

credito = wbdata.get_dataframe(
    {"FS.AST.PRVT.GD.ZS": "CREDIT (% GDP)"},
    country=paises,
    date=(ano_inicial, ano_final)
)

tabela_credito = credito.reset_index()

tabela_credito = tabela_credito.rename(columns={
    "country": "COUNTRY",
    "date": "YEAR"
})

if not tabela_credito.empty:
    tabela_credito["YEAR"] = pd.to_numeric(
        tabela_credito["YEAR"], errors="coerce"
    )
    tabela_credito = tabela_credito.sort_values(
        ["COUNTRY", "YEAR"]
    )


# 4. PESQUISAR INDICADORES DE DERIVATIVOS

indicadores_derivativos = wbdata.get_indicators(
    query="derivative"
)

# Criar a terceira tabela de dados financeiros
# apenas se houver um indicador apropriado disponível.
#
# A busca por nome não garante que exista uma série
# anual comparável para os quatro países.

tabela_derivativos = pd.DataFrame()

if indicadores_derivativos:
    tabela_indicadores = pd.DataFrame(
        indicadores_derivativos
    )

    colunas = [
        coluna for coluna in ["id", "name"]
        if coluna in tabela_indicadores.columns
    ]

    if colunas:
        tabela_derivativos = tabela_indicadores[colunas].copy()


# 5. BUSCAR OS DADOS DO PIB

pib = wbdata.get_dataframe(
    {"NY.GDP.MKTP.CD": "GDP (US$)"},
    country=paises,
    date=(ano_inicial, ano_final)
)

tabela_pib = pib.reset_index()

tabela_pib = tabela_pib.rename(columns={
    "country": "COUNTRY",
    "date": "YEAR"
})

if not tabela_pib.empty:
    tabela_pib["YEAR"] = pd.to_numeric(
        tabela_pib["YEAR"], errors="coerce"
    )

    tabela_pib["GDP (US$ BILLIONS)"] = (
        tabela_pib["GDP (US$)"] / 1_000_000_000
    )

    tabela_pib = tabela_pib[
        ["COUNTRY", "YEAR", "GDP (US$ BILLIONS)"]
    ]

    tabela_pib = tabela_pib.sort_values(
        ["COUNTRY", "YEAR"]
    )


# 6. MOSTRAR AS TRÊS TABELAS

display(Markdown("## 1. CREDIT — WORLD BANK"))

if tabela_credito.empty:
    print("No credit data found.")
else:
    display(
        tabela_credito.style
        .format({"CREDIT (% GDP)": "{:,.2f}"})
        .hide(axis="index")
    )


display(Markdown("## 2. DERIVATIVES — INDICATOR SEARCH"))

if tabela_derivativos.empty:
    print(
        "No indicator containing 'derivative' was found "
        "in the World Bank indicator search."
    )
else:
    display(tabela_derivativos)


display(Markdown("## 3. GDP — WORLD BANK"))

if tabela_pib.empty:
    print("No GDP data found.")
else:
    display(
        tabela_pib.style
        .format({"GDP (US$ BILLIONS)": "{:,.2f}"})
        .hide(axis="index")
    )


## 1. CREDIT — WORLD BANK

COUNTRY,YEAR,CREDIT (% GDP)
China,2000,109.99
China,2001,108.77
China,2002,115.98
China,2003,123.91
China,2004,116.91
China,2005,110.32
China,2006,107.62
China,2007,104.21
China,2008,100.41
China,2009,122.30


## 2. DERIVATIVES — INDICATOR SEARCH

No indicator containing 'derivative' was found in the World Bank indicator search.


## 3. GDP — WORLD BANK

COUNTRY,YEAR,GDP (US$ BILLIONS)
China,2000,"1,223.75"
China,2001,"1,355.04"
China,2002,"1,489.82"
China,2003,"1,683.90"
China,2004,"1,984.20"
China,2005,"2,317.55"
China,2006,"2,791.50"
China,2007,"3,604.06"
China,2008,"4,667.35"
China,2009,"5,189.58"
